# 0929 — SDS-PAGE annotation and analysis: second pair of expression-test gels

Annotates the two 2026-09-29 Coomassie gels and asks whether any construct expressed. Same pipeline
as [`0915_gel_annotation.ipynb`](0915_gel_annotation.ipynb); the differences are listed in §2.

## Verdict: two candidate bands, neither assigned yet

| | |
|---|---|
| **CDS12_WT** (gel 1, lanes 2–7) | **IPTG-only band at 40–43 kDa, on the predicted 41.8 kDa.** Strong in the induced raw lane (prominence 0.106), weak at the same position in the uninduced raw lane (0.029) even though that lane carries *more* total protein. First induction-dependent signal in this project. |
| **ANC_Node1 / e1** (gel 2, lanes 8–11) | **Strong, clean band at 45–50 kDa in all four lanes**, induced = uninduced, scaling with dilution. It sits 7–19 % above the predicted 41.9 kDa — at or beyond the edge of the ±9 % window. Candidate, not assigned. |
| **CDS11_WT** (gel 1, lanes 8–11) | **Nothing at 72 kDa.** The two IPTG lanes carry a thin band at 49–51 kDa that the two uninduced lanes lack. No raw lane was run. |
| **ANC_Node281 / e281** (gel 2, lanes 2–7) | **No band at the predicted 42.3 kDa.** Raw lanes are overloaded smears, dilution lanes are blank. |

Both gels are well run this time: every lane is loaded, both ladders resolve all 10 rungs, and the
asymmetric loading pattern fixes the left–right orientation (§1). That makes this the first usable
data set. It is still not proof of expression, for three reasons:

1. **No empty-vector or untransformed-BL21 lane — UNDONE, second time.** Without it a band cannot be
   assigned to a construct rather than to the host.
2. **An unexplained ~48–50 kDa band.** It appears in CDS12 uninduced (lanes 5–7, 45–50 kDa) and CDS11 IPTG
   (lanes 8–9) on gel 1, and the e1 band on gel 2 runs within 3 kDa of it. 0915 saw a band at
   ~50–52 kDa in its CDS12 lanes too. It may be one recurring species.
3. **Single replicate, mass only.** Coomassie reports mass, not identity.

## What to run next

1. **Anti-His western or Ni-NTA pulldown — UNDONE.** Both candidate bands are strong enough to test
   directly. All six constructs carry the N-terminal His6
   ([`0910_construct_protein_mw.ipynb`](0910_construct_protein_mw.ipynb)).
2. **Empty-vector / untransformed-BL21 lane — UNDONE.** Decides reason 2 in one gel.
3. **Load 1:2–1:5 instead of raw / 1:10 / 1:20.** Raw smears; 1:10 and 1:20 fall below detection for
   everything except a dominant band.
4. **Run the missing raw lanes — UNDONE**: CDS11 and e1 were loaded at 1:10 and 1:20 only.
5. **Photograph on a clean white surface.** The paper texture under the tray sets the noise floor (§6).

## Inputs

| File | Original name in `sds_gel_0929/` | What |
|---|---|---|
| `data/assays/0929_sds_page/cds12-cds11.jpeg` | `cds.jpeg` | gel 1 — CDS12_WT / CDS11_WT |
| `data/assays/0929_sds_page/e281-e1.jpeg` | `e.jpeg` | gel 2 — CDS12_ANC_Node281 / Node1 |
| `data/assays/0929_sds_page/cds12-cds11_label.jpeg` | `cds_label.jpeg` | printed loading map, gel 1 |
| `data/assays/0929_sds_page/e281-e1_label.jpeg` | `e_label.jpeg` | printed loading map, gel 2 |

Phone photos of the stained gel in a tray on a light box, over textured paper. Bands are visible
without processing, unlike 0915. Predicted masses come from `0910_construct_protein_mw.ipynb`.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps
from matplotlib.patches import Polygon
from scipy.ndimage import gaussian_filter, gaussian_filter1d, percentile_filter
from scipy.signal import find_peaks

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
IMG  = ROOT/"data/assays/0929_sds_page"
OUT  = ROOT/"code/output/0929_gel_annotation"; OUT.mkdir(parents=True, exist_ok=True)

# Predicted masses (0910 notebook): includes pET-28a's 21 aa MGSSHHHHHHSSGLVPRGSHM leader
PREDICTED = {"CDS12": 41.84, "CDS11": 72.02, "e281": 42.27, "e1": 41.91}

_img = {}
def load(fn):
    "Phone photos carry an EXIF rotation; without transpose the gel comes out landscape."
    if fn not in _img:
        _img[fn] = np.asarray(ImageOps.exif_transpose(Image.open(IMG/fn)).convert("RGB")).astype(float)/255.
    return _img[fn]

## 1. Loading plan, transcribed from the two label sheets

Printed map, "SDS-PAGE Loading Map · 2 gels × 12 wells · Coomassie R-250 · Ladder in wells 1 and 12
(7 µL) · Raw = undiluted extract".

| Lane | Gel 1 (`cds12-cds11`) | Gel 2 (`e281-e1`) |
|---|---|---|
| 1 | Ladder | Ladder |
| 2–4 | CDS12 IPTG: raw, 1:10, 1:20 | e281 IPTG: raw, 1:10, 1:20 |
| 5–7 | CDS12 Non-Ind.: raw, 1:10, 1:20 | e281 Non-Ind.: raw, 1:10, 1:20 |
| 8–9 | CDS11 IPTG: 1:10, 1:20 | e1 IPTG: 1:10, 1:20 |
| 10–11 | CDS11 Non-Ind.: 1:10, 1:20 | e1 Non-Ind.: 1:10, 1:20 |
| 12 | Ladder | Ladder |

> ⚠️ **Ambiguities, surfaced before anything is measured.**
> 1. **The sheets are not filled in.** Date, gel %, voltage/time and every sample volume (`____ µL`)
>    are blank. The date 2026-09-29 is taken from the photo EXIF (09:10–09:20). Loaded volume is unknown.
> 2. **"Extract" is not defined.** Whole-cell, or cleared lysate? If cleared, an insoluble product
>    would be invisible on these gels, and "no band" for e281 and CDS11 would not mean "no expression".
> 3. **No raw lane for CDS11 or e1**, and no control lane of any kind.
> 4. A "BCA Dilution Plate" sheet is visible under the gel 1 label. The BCA readings are not in the repo.

**Two 0915 caveats are resolved here.** Orientation is no longer assumed: the heavy raw lanes sit
in wells 2 and 5 counted from the left, and a flipped gel would put them in wells 8 and 11. Lane
numbers on the sheet match the gel: 12 wells, 12 lanes, ladders outermost.

## 2. Background correction — what changed from 0915, and why

0915 divided by a 70 px Gaussian estimate of the illumination. That was right for thin bands on a
washed-out photo. Here a raw lane is a 200 px wide smear running the whole height of the gel, and a
70 px Gaussian would treat the smear itself as background and divide it away.

So the background is estimated as the **80th percentile in a 400 px window**, then smoothed. A window
wider than a lane always contains clear gel, and the 80th percentile picks that clear gel rather
than the lane. Measurements use optical density, `OD = −log10(I / background)`, so a band twice as
dark counts twice. **Totals here are therefore not numerically comparable with the 0915 table.**

Every figure is again produced twice:

- `_uniform` — a single linear contrast stretch, nothing else. Publication-safe under iGEM's image
  rules; use this one on the wiki.
- `_flattened` — background-divided, for visibility, disclosed in the caption.

No local, per-lane or per-band edits anywhere.

In [2]:
# gel body crops: below the wells, above the dye front
BOX = {"cds12-cds11.jpeg": (500, 3640, 1600, 4150),
       "e281-e1.jpeg":     (580, 3650, 1580, 4060)}

def crop(fn):
    x0,x1,y0,y1 = BOX[fn]
    return load(fn)[y0:y1, x0:x1]

def background(ch, win=400, pct=80, ds=8):
    "Clear-gel level: 80th percentile in a window wider than a lane, computed on a /8 thumbnail."
    small = gaussian_filter(ch, ds)[::ds, ::ds]
    bg    = gaussian_filter(percentile_filter(small, pct, size=win//ds), win/ds/4)
    im    = Image.fromarray(bg.astype(np.float32)).resize(ch.shape[::-1], Image.BILINEAR)
    return np.clip(np.asarray(im), 1e-3, None)

_od = {}
def od(fn):
    if fn not in _od:
        lum = crop(fn).mean(2)
        _od[fn] = -np.log10(np.clip(lum/background(lum), 1e-3, None))   # ~0 = clear gel, >0 = band
    return _od[fn]

## 3. Which ladder is this?

Same test as 0915, adapted to a stained gel. The whole gel is blue here, so absolute colour is
useless; each band's red and green excess is taken **relative to the median band in its own lane**.

In [3]:
ROUGH_X = {"cds12-cds11.jpeg": (305, 2845), "e281-e1.jpeg": (298, 2808)}

def ladder_bands(fn, w=45):
    g, o = crop(fn), od(fn); rows = []
    for tag, lx in zip("LR", ROUGH_X[fn]):
        prof  = gaussian_filter1d(o[:, lx-w:lx+w].mean(1), 6)
        bp, _ = find_peaks(prof, prominence=0.04, distance=40)
        rgb   = np.array([g[p-8:p+8, lx-w:lx+w].reshape(-1, 3).mean(0) for p in bp])
        red   = rgb[:,0] - (rgb[:,1] + rgb[:,2])/2
        green = rgb[:,1] - (rgb[:,0] + rgb[:,2])/2
        red, green = red - np.median(red), green - np.median(green)
        for i, p in enumerate(bp):
            rows.append(dict(gel=fn[:-5], lane=tag, band=i+1, y=int(p), od=round(float(prof[p]), 3),
                             colour="RED" if red[i] > .10 else ("GREEN" if green[i] > .08 else "blue"),
                             red_excess=round(float(red[i]), 3), green_excess=round(float(green[i]), 3)))
    return pd.DataFrame(rows)

bands = pd.concat([ladder_bands(f) for f in BOX])
print(bands.groupby(["gel","lane"]).size().to_string(), "\n")
print(bands[bands.colour != "blue"].to_string(index=False))

gel          lane
cds12-cds11  L       10
             R       10
e281-e1      L       10
             R       10 

        gel lane  band    y    od colour  red_excess  green_excess
cds12-cds11    L     4 1269 0.207    RED       0.246        -0.073
cds12-cds11    L    10 2487 0.101  GREEN       0.061         0.155
cds12-cds11    R     4 1242 0.235    RED       0.226        -0.067
cds12-cds11    R    10 2481 0.078  GREEN       0.043         0.118
    e281-e1    L     4 1202 0.237    RED       0.251        -0.065
    e281-e1    L    10 2413 0.114  GREEN       0.028         0.174
    e281-e1    R     4 1191 0.289    RED       0.211        -0.092
    e281-e1    R    10 2384 0.186  GREEN       0.015         0.172


All four ladder lanes: **10 bands, the 4th from the top red, the lowest green.** That is the same
**Thermo PageRuler Prestained Protein Ladder** identified and confirmed against the product sheet
in 0915 — 170, 130, 100, **70 (orange)**, 55, 40, 35, 25, 15, **10 (green)** kDa.

In [4]:
LADDER_MW = np.array([170,130,100,70,55,40,35,25,15,10], float)
LOGMW     = np.log10(LADDER_MW)

# Band y-positions as detected above, left and right ladder lane of each gel.
CAL = {"cds12-cds11.jpeg": dict(yL=[728,822,1064,1269,1542,1781,1991,2127,2367,2487],
                                yR=[704,797,1033,1242,1501,1754,1968,2118,2359,2481]),
       "e281-e1.jpeg":     dict(yL=[672,759,994,1202,1460,1707,1930,2070,2307,2413],
                                yR=[664,754,981,1191,1431,1694,1909,2062,2288,2384])}

for fn in BOX:                                  # the hard-coded values must be what was detected
    for tag, key in zip("LR", ("yL", "yR")):
        found = bands[(bands.gel == fn[:-5]) & (bands.lane == tag)].y.tolist()
        assert found == CAL[fn][key], (fn, tag, found)
print("CAL matches the detected bands on all four ladder lanes")

CAL matches the detected bands on all four ladder lanes


## 4. Calibration and its precision

0915 used a single left-to-right tilt. Here the two ladders differ by 30 px at the top and 6 px at
the bottom, so each rung is interpolated between its own left and right position instead.

**Two calibrations are kept side by side**, because they disagree where it matters:

- **cubic** — `log10(MW)` against migration, cubic fit. The 0915 method, kept for comparability.
- **rung-to-rung** — log-linear interpolation between the two flanking rungs. Exact at the rungs.

The rungs of this ladder are not evenly spaced in `log10(MW)`, so the cubic cannot pass through
them. Its residual pattern is the same on all four lanes and the same as in 0915.

In [5]:
rows = []
for fn in BOX:
    for tag, key in zip(("left","right"), ("yL","yR")):
        y = np.array(CAL[fn][key], float); c3 = np.polyfit(y, LOGMW, 3)
        fit = 10**np.polyval(c3, y)
        rows.append(dict(lane=f"{fn[:-5]} {tag}",
                         max_err=f"{np.abs(fit/LADDER_MW-1).max()*100:.0f}%",
                         **{f"{m:.0f}": round(f/m, 2) for m, f in zip(LADDER_MW, fit)}))
print("cubic fit / nominal mass, per rung")
print(pd.DataFrame(rows).to_string(index=False))
print("\nSame error pattern on all four lanes -> a property of the ladder on this gel, not noise."
      "\nWorking precision stays +/-9%, i.e. +/-4 kDa on a 42 kDa target."
      "\nBetween the 55 and 40 kDa rungs the cubic reads 2-3 kDa higher than rung-to-rung.")

cubic fit / nominal mass, per rung
             lane max_err  170  130  100   70   55   40   35   25   15   10
 cds12-cds11 left      8% 0.98 1.06 0.92 1.03 1.00 1.07 0.92 1.02 0.98 1.02
cds12-cds11 right      8% 0.99 1.06 0.92 1.02 1.01 1.07 0.93 1.01 0.98 1.02
     e281-e1 left      8% 0.98 1.06 0.92 1.02 1.01 1.08 0.92 1.01 0.96 1.04
    e281-e1 right      9% 0.99 1.05 0.92 1.01 1.02 1.09 0.94 1.00 0.95 1.05

Same error pattern on all four lanes -> a property of the ladder on this gel, not noise.
Working precision stays +/-9%, i.e. +/-4 kDa on a 42 kDa target.
Between the 55 and 40 kDa rungs the cubic reads 2-3 kDa higher than rung-to-rung.


## 5. Lane grid

Twelve lanes, evenly spaced between the two ladder lanes. The gel flares outward towards the
bottom, so each ladder's centre line is fitted as a straight line in `y` and the ten sample lanes
are interpolated between the two lines at every height.

The check is independent of the ladders: band centres measured directly on sample lanes.

In [6]:
NL = 12
# centre line of each ladder lane, x at the top and bottom edge of the crop (fitted below)
LANE_X = {"cds12-cds11.jpeg": dict(top=(326, 2807), bottom=(270, 2869)),
          "e281-e1.jpeg":     dict(top=(319, 2780), bottom=(249, 2820))}

def lane_x(fn, i, y):
    H = od(fn).shape[0]; (l0, r0), (l1, r1) = LANE_X[fn]["top"], LANE_X[fn]["bottom"]
    xl, xr = l0 + (l1-l0)*y/H, r0 + (r1-r0)*y/H
    return xl + (xr-xl)*i/(NL-1)

def pitch(fn, y):
    return (lane_x(fn, NL-1, y) - lane_x(fn, 0, y))/(NL-1)

def band_centre(fn, y, x, half):
    "Horizontal centre of a band at row y: midpoint of its extent at half height."
    row  = gaussian_filter1d(od(fn)[y-12:y+12].mean(0), 6)[int(x-half):int(x+half)]
    base = np.percentile(row, 10); idx = np.where(row - base > 0.5*(row.max() - base))[0]
    return (idx.min() + idx.max())/2 + int(x-half), idx.max() - idx.min()

for fn in BOX:
    H = od(fn).shape[0]
    for tag, key, rough in zip("LR", ("yL","yR"), ROUGH_X[fn]):
        pts = np.array([(y, *band_centre(fn, y, rough, 170)) for y in CAL[fn][key]], float)
        ok  = np.abs(pts[:,2]/np.median(pts[:,2]) - 1) < 0.12      # drop bands merged with a neighbour
        c   = np.polyfit(pts[ok,0], pts[ok,1], 1)
        print(f"{fn:18s} ladder {tag}: x(top) {np.polyval(c,0):.0f}  x(bottom) {np.polyval(c,H):.0f}"
              f"   from {ok.sum()} clean rungs")
    print(f"{'':18s} lane pitch {pitch(fn, H/2):.0f} px")

CHECK = {"cds12-cds11.jpeg": [(5, 1680), (6, 1631), (8, 1612)],
         "e281-e1.jpeg":     [(8, 1579), (9, 1564), (10, 1605), (11, 1592)]}
print("\nindependent check on sample bands")
for fn, lst in CHECK.items():
    for lane, y in lst:
        x = lane_x(fn, lane-1, y); obs, _ = band_centre(fn, y, x, pitch(fn, y)*.6)
        print(f"  {fn:18s} lane {lane:2d}: grid {x:6.0f}   observed {obs:6.0f}   off by {obs-x:+4.0f} px")

cds12-cds11.jpeg   ladder L: x(top) 326  x(bottom) 270   from 5 clean rungs
cds12-cds11.jpeg   ladder R: x(top) 2807  x(bottom) 2869   from 7 clean rungs
                   lane pitch 231 px
e281-e1.jpeg       ladder L: x(top) 319  x(bottom) 249   from 3 clean rungs
e281-e1.jpeg       ladder R: x(top) 2780  x(bottom) 2820   from 8 clean rungs
                   lane pitch 229 px

independent check on sample bands
  cds12-cds11.jpeg   lane  5: grid   1220   observed   1234   off by  +15 px
  cds12-cds11.jpeg   lane  6: grid   1452   observed   1432   off by  -21 px
  cds12-cds11.jpeg   lane  8: grid   1917   observed   1892   off by  -24 px
  e281-e1.jpeg       lane  8: grid   1885   observed   1888   off by   +2 px
  e281-e1.jpeg       lane  9: grid   2115   observed   2114   off by   -1 px
  e281-e1.jpeg       lane 10: grid   2345   observed   2341   off by   -4 px
  e281-e1.jpeg       lane 11: grid   2576   observed   2564   off by  -12 px


The grid lands within about 25 px of the observed band centres, against a lane pitch of 230 px and
a measuring strip of ±63 px. Faint bands give the larger offsets because their edges are noisy.

## 6. Quantification

Per lane, in optical density:

- **total** — integrated OD over the lane. Total protein loaded, nothing more.
- **contrast at the predicted mass** — peak height inside a ±8 % window around the predicted mass,
  above the median of the flanking gel. Same definition as 0915.
- **strongest band, 30–100 kDa** — new. Its position, its apparent mass by both calibrations, its
  prominence, and its volume (OD summed over ±45 rows, flanks subtracted).

**Noise floor.** Lanes with nothing visible still produce peaks of prominence up to 0.05, because
the paper under the tray is textured. A band counts as `detected` only at prominence ≥ 0.055.

In [7]:
LANES = {
 "cds12-cds11.jpeg": ["P.L","CDS12 IPTG raw","CDS12 IPTG 1:10","CDS12 IPTG 1:20",
                      "CDS12 non raw","CDS12 non 1:10","CDS12 non 1:20",
                      "CDS11 IPTG 1:10","CDS11 IPTG 1:20","CDS11 non 1:10","CDS11 non 1:20","P.L"],
 "e281-e1.jpeg":     ["P.L","e281 IPTG raw","e281 IPTG 1:10","e281 IPTG 1:20",
                      "e281 non raw","e281 non 1:10","e281 non 1:20",
                      "e1 IPTG 1:10","e1 IPTG 1:20","e1 non 1:10","e1 non 1:20","P.L"],
}
SEARCH, FLOOR = (30, 100), 0.055

def rungs(fn, f):
    "Rung positions at lane fraction f (0 = left ladder, 1 = right ladder)."
    yL, yR = np.array(CAL[fn]["yL"], float), np.array(CAL[fn]["yR"], float)
    return yL + f*(yR - yL)

def y_of(fn, mw, f):         return float(np.polyval(np.polyfit(LOGMW, rungs(fn, f), 3), np.log10(mw)))
def mw_cubic(fn, y, f):      return float(10**np.polyval(np.polyfit(rungs(fn, f), LOGMW, 3), y))
def mw_piecewise(fn, y, f):  return float(10**np.interp(y, rungs(fn, f), LOGMW))

def profile(fn, i, width=.275, shift=0.):
    "Mean OD across the lane, row by row, following the lane's centre line."
    o = od(fn)
    col = np.array([o[y, int(lane_x(fn,i,y) + pitch(fn,y)*(shift-width)):
                         int(lane_x(fn,i,y) + pitch(fn,y)*(shift+width))].mean()
                    for y in range(o.shape[0])])
    return gaussian_filter1d(col, 5)

def volume(fn, i, p, hw=45, gap=20, fl=70):
    o = od(fn); x = lane_x(fn, i, p); w = pitch(fn, p)*.45
    sl = o[:, int(x-w):int(x+w)].mean(1)
    base = np.median(np.r_[sl[p-hw-gap-fl:p-hw-gap], sl[p+hw+gap:p+hw+gap+fl]])
    return float((sl[p-hw:p+hw] - base).sum())

def analyse(fn):
    H = od(fn).shape[0]; zone = slice(int(.02*H), int(.99*H)); rows = []
    for i, nm in enumerate(LANES[fn]):
        f = i/(NL-1); col = profile(fn, i)
        rec = dict(lane=i+1, sample=nm, total=round(float(col[zone].sum()), 1))
        if nm != "P.L":
            mw = PREDICTED[nm.split()[0]]
            ye = y_of(fn, mw, f); wd = abs(y_of(fn, mw*1.08, f) - y_of(fn, mw*0.92, f))/2
            a, b = int(ye-wd), int(ye+wd)
            flank = np.r_[col[max(0, a-3*int(wd)):a], col[b:b+3*int(wd)]]
            rec |= dict(predicted_kDa=mw, y_predicted=int(ye),
                        contrast=round(float(col[a:b].max() - np.median(flank)), 4))
            lo, hi = y_of(fn, SEARCH[1], f), y_of(fn, SEARCH[0], f)
            pk, pr = find_peaks(col, prominence=0.004, distance=15)
            p, q = max([(p, q) for p, q in zip(pk, pr["prominences"]) if lo < p < hi],
                       key=lambda t: t[1])
            rec |= dict(band_y=int(p), band_kDa_cubic=round(mw_cubic(fn, p, f), 1),
                        band_kDa_piecewise=round(mw_piecewise(fn, p, f), 1),
                        band_prominence=round(float(q), 4),
                        band_volume=round(volume(fn, i, int(p)), 2), detected=bool(q >= FLOOR))
        rows.append(rec)
    df = pd.DataFrame(rows)
    df["total_vs_ladder"] = (df["total"]/df.loc[df["sample"].eq("P.L"), "total"].mean()).round(2)
    return df

pd.set_option("display.width", 250)
tables = {fn: analyse(fn) for fn in BOX}
for fn in BOX:
    H = od(fn).shape[0]; zone = slice(int(.02*H), int(.99*H))
    gaps = [profile(fn, i, width=.05, shift=.5)[zone].sum() for i in range(1, NL-2)]
    print(f"\n=== {fn} ===   blank-gel floor for `total`, from the {len(gaps)} gaps between sample "
          f"lanes: {np.min(gaps):.0f}-{np.max(gaps):.0f}")
    print(tables[fn].to_string(index=False))


=== cds12-cds11.jpeg ===   blank-gel floor for `total`, from the 9 gaps between sample lanes: 16-38
 lane          sample  total  predicted_kDa  y_predicted  contrast  band_y  band_kDa_cubic  band_kDa_piecewise  band_prominence  band_volume detected  total_vs_ladder
    1             P.L  106.7            NaN          NaN       NaN     NaN             NaN                 NaN              NaN          NaN      NaN             0.89
    2  CDS12 IPTG raw  101.6          41.84       1757.0    0.0610  1776.0            43.0                40.1           0.1061         2.83     True             0.85
    3 CDS12 IPTG 1:10   38.9          41.84       1755.0    0.0248  1737.0            44.8                42.1           0.0350         0.59    False             0.33
    4 CDS12 IPTG 1:20   25.0          41.84       1752.0    0.0150  1767.0            43.2                40.3           0.0322         0.49    False             0.21
    5   CDS12 non raw  131.1          41.84       1750.0    0.03


=== e281-e1.jpeg ===   blank-gel floor for `total`, from the 9 gaps between sample lanes: 14-48
 lane         sample  total  predicted_kDa  y_predicted  contrast  band_y  band_kDa_cubic  band_kDa_piecewise  band_prominence  band_volume detected  total_vs_ladder
    1            P.L  106.5            NaN          NaN       NaN     NaN             NaN                 NaN              NaN          NaN      NaN             0.93
    2  e281 IPTG raw  150.0          42.27       1681.0    0.0033  1526.0            52.1                50.4           0.0709         2.22     True             1.31
    3 e281 IPTG 1:10   46.1          42.27       1680.0   -0.0018  1311.0            63.6                63.0           0.0492         0.37    False             0.40
    4 e281 IPTG 1:20   27.0          42.27       1678.0    0.0033  1367.0            60.2                59.6           0.0241        -0.10    False             0.24
    5   e281 non raw  151.9          42.27       1677.0    0.0107  1471.0

### Reading those numbers

**Loading.** Raw lanes carry 0.85–1.32× the ladder lane. The 1:10 and 1:20 lanes read 25–55, which
is inside the blank-gel floor (14–48): a dilution lane shows only what dominates its sample. Raw
lanes are overloaded — they smear, and gel 2 lane 5 pinches — so band positions in raw lanes are
less reliable than in dilution lanes.

**Gel 1, CDS12 — induction-dependent band on the predicted mass.**

| contrast at 41.8 kDa | raw | 1:10 | 1:20 |
|---|---|---|---|
| IPTG | **0.061** | 0.025 | 0.015 |
| uninduced | 0.032 | 0.019 | 0.003 |

Induced is higher at all three dilutions, but only the raw pair clears the noise floor. The
strongest band of the induced raw lane is at 40–43 kDa. The strongest band of the uninduced raw
lane is at 45–47 kDa, a different band 96 px higher up the gel, and that lane carries more total
protein (131 against 102). So the difference is not loading. The induced 1:10 lane shows a faint
band by eye at the height of the raw band; at prominence 0.035 it is below the floor and is not
counted.

**Gel 1, the 48–50 kDa band.** Thin and sharp in lanes 6–9: CDS12 uninduced 1:10 and 1:20, CDS11
IPTG 1:10 and 1:20. Prominence alternates 0.059 / 0.041 / 0.076 / 0.056, as a 1:10 / 1:20 series
should (lane 7 is visible by eye but below the floor). It is absent from the CDS12 IPTG dilutions
and from the CDS11 uninduced lanes. **That pattern follows neither construct nor induction; it
follows four physically adjacent wells.** A host protein whose level varies between cultures, a
truncated product, and a loading mix-up across wells 6–9 all fit. The gel cannot tell them apart.

**Gel 1, CDS11.** Contrast at 72 kDa is −0.002 to 0.019: nothing. With no raw lane, a weakly
expressed product would not show at 1:10 either.

**Gel 2, e281.** Contrast at 42.3 kDa is 0.003 induced and 0.011 uninduced in the raw lanes. The
strongest bands there are at 50–55 kDa, in both lanes. No induction signal.

**Gel 2, e1 — the strongest band on either gel.** Prominence 0.10–0.11 in all four lanes. Band
volume 3.1 / 1.9 induced and 3.1 / 2.7 uninduced for 1:10 / 1:20. A 1:10 e1 lane holds as much in
this one band as the CDS12 induced **raw** lane holds in its 40–43 kDa band (2.8), so undiluted e1
extract holds roughly ten times more. Nothing else is visible in those lanes: one species dominates
the sample.

| e1 band | lane 8 | lane 9 | lane 10 | lane 11 |
|---|---|---|---|---|
| rung-to-rung, kDa | 46.4 | 47.1 | 44.7 | 45.4 |
| cubic, kDa | 49.2 | 49.8 | 47.8 | 48.4 |

It sits about midway between the 55 and 40 kDa rungs, against a predicted 41.9 kDa.

- *For* the assignment: His-tagged proteins often run a few kDa above their calculated mass, and
  equal intensity with and without IPTG is what leaky or auto-induced pET expression looks like.
- *Against*: CDS12_WT is 0.07 kDa lighter than e1, and its candidate band on gel 1 runs **on** the
  predicted mass, not above it. If both assignments were right, two proteins of the same mass would
  run about 5 kDa apart. Sequence composition can shift migration, but not for free. **At most one
  of the two assignments is the simple one.**

## 7. Annotated figures

Eight files, JPEG q92 at dpi 120: each gel × {`uniform`, `flattened`} × {`gray`, `color`}.

- **`uniform`** — one linear contrast stretch across the whole frame, nothing else. Publication-safe
  under iGEM's image rules; use it on the wiki.
- **`flattened`** — divided by the §2 background so the faint lanes are visible. A non-uniform
  correction, disclosed in the caption. Use it to read the gel, not to publish it.
- **`gray`** — luminance only. Cleanest for band position.
- **`color`** — RGB kept. The `uniform` colour figure is the photograph with one stretch applied to
  all three channels. It is **not** white-balanced as in 0915: the gel background is stained blue,
  and balancing on it turns the paper yellow. The `flattened` colour figure divides each channel by
  its own background and lifts saturation globally. The ladder's orange 70 kDa and green 10 kDa
  rungs show as themselves in both.

The **red box** is as in 0915: it spans the construct's lanes and its height is the ±9 % calibration
window of the cubic fit. It follows the ladder tilt and the flare of the lanes.

**New: black arrowheads.** One per sample lane whose strongest 30–100 kDa band clears the noise
floor, labelled with its apparent mass as `rung-to-rung – cubic`. In an overloaded raw lane the
arrowhead marks the strongest peak of the lysate, which need not be the construct.

In [8]:
REF  = {70: "#E8590C", 10: "#2F9E44"}                    # the two coloured reference bands
ORANGE, RED, INK, GREY = "#C2410C", "#DC2626", "#111111", "#6B7280"
SAT  = 1.6        # global saturation multiplier, flattened colour set only
TOL  = 1.09       # the +/-9% calibration precision, used as the box height

# lanes are 0-based here; `tag` = the two lanes the box label is centred under
G = {"cds12-cds11.jpeg": dict(gel="Gel 1", blocks=[
        dict(name="CDS12_WT", mw=41.84, col="#1D4ED8", lanes=(1, 6),
             groups=[("IPTG", 1, 3), ("non", 4, 6)], tag=(2, 3)),
        dict(name="CDS11_WT", mw=72.02, col="#7C3AED", lanes=(7, 10),
             groups=[("IPTG", 7, 8), ("non", 9, 10)], tag=(8, 9))]),
     "e281-e1.jpeg": dict(gel="Gel 2", blocks=[
        dict(name="ANC_Node281", mw=42.27, col="#1D4ED8", lanes=(1, 6),
             groups=[("IPTG", 1, 3), ("non", 4, 6)], tag=(2, 3)),
        dict(name="ANC_Node1",   mw=41.91, col="#7C3AED", lanes=(7, 10),
             groups=[("IPTG", 7, 8), ("non", 9, 10)], tag=(8, 9))])}
DIL = [None, "raw", "1:10", "1:20", "raw", "1:10", "1:20", "1:10", "1:20", "1:10", "1:20", None]


def gelimg(fn, corr, palette):
    rgb = crop(fn)
    if palette == "gray":
        lum = rgb.mean(2)
        img = lum/background(lum) if corr == "flattened" else lum
        lo, hi = np.percentile(img, [0.3, 99.7] if corr == "flattened" else [0.05, 99.9])
        return np.clip((img - lo)/(hi - lo), 0, 1)
    if corr == "uniform":
        lo, hi = np.percentile(rgb, [1, 99.5])
        return np.clip((rgb - lo)/(hi - lo), 0, 1)
    out = np.dstack([rgb[..., k]/background(rgb[..., k]) for k in range(3)])
    lo, hi = np.percentile(out, [0.5, 99.8])
    out = np.clip((out - lo)/(hi - lo), 0, 1)
    m = out.mean(2, keepdims=True)
    return np.clip(m + (out - m)*SAT, 0, 1)


def draw(fn, corr, palette, OUT, tables):
    c = G[fn]; v = gelimg(fn, corr, palette); H, W = v.shape[:2]
    lx = np.array([lane_x(fn, i, 0) for i in range(NL)]); step = pitch(fn, 0)
    HEAD, SIDE = 0.36*H, 0.115*W
    gray = palette == "gray"

    fig = plt.figure(figsize=(16, 16*(H + HEAD)/(W + 2*SIDE)))
    ax  = fig.add_axes([0, 0.042, 1, 0.888])
    ax.imshow(v, cmap="gray" if gray else None, vmin=0 if gray else None,
              vmax=1 if gray else None, aspect="equal", zorder=1)

    for f, sgn in [(0., -1), (1., 1)]:
        xtxt   = -0.070*W if sgn < 0 else W + 0.070*W
        t0, t1 = (-0.050*W, -0.012*W) if sgn < 0 else (W + 0.012*W, W + 0.050*W)
        ys = rungs(fn, f)
        for m, y in zip(LADDER_MW, ys):
            lane_edge = lane_x(fn, 0 if sgn < 0 else NL-1, y) - sgn*pitch(fn, y)*0.46
            col = REF.get(int(m), ORANGE); big = int(m) in REF
            ax.plot([t0, t1], [y, y], color=col, lw=3.4 if big else 2.4,
                    solid_capstyle="butt", clip_on=False, zorder=6)
            ax.plot([t1, lane_edge], [y, y], color=col, lw=1.1, alpha=.50, clip_on=False, zorder=5)
            ax.text(xtxt, y, f"{m:.0f}", color=col, fontsize=17 if big else 15,
                    fontweight="bold", ha="center", va="center", clip_on=False, zorder=7)
        ax.text(xtxt, ys[0] - 0.055*H, "kDa", color=ORANGE, fontsize=14, style="italic",
                fontweight="bold", ha="center", va="center", clip_on=False, zorder=7)

    ax.add_patch(plt.Rectangle((-SIDE, -HEAD), W + 2*SIDE, HEAD, fc="white", ec="none", zorder=2))
    Y = dict(name=-.300, pred=-.252, brk=-.216, cond=-.158, cbrk=-.131, dil=-.083, num=-.034)
    for b in c["blocks"]:
        i0, i1 = b["lanes"]; a, z = lx[i0] - step*.44, lx[i1] + step*.44
        ax.plot([a, a, z, z], [Y["brk"]*H + .020*H, Y["brk"]*H, Y["brk"]*H, Y["brk"]*H + .020*H],
                color=b["col"], lw=3.0, zorder=6)
        ax.text((a + z)/2, Y["name"]*H, b["name"], color=b["col"], fontsize=23, fontweight="bold",
                ha="center", va="center", zorder=6)
        ax.text((a + z)/2, Y["pred"]*H, f"predicted {b['mw']:.1f} kDa", color=RED, fontsize=16,
                fontweight="bold", ha="center", va="center", zorder=6)
        for cond, j0, j1 in b["groups"]:
            a2, z2 = lx[j0] - step*.40, lx[j1] + step*.40
            ax.plot([a2, a2, z2, z2], [Y["cbrk"]*H + .016*H, Y["cbrk"]*H, Y["cbrk"]*H,
                                       Y["cbrk"]*H + .016*H], color=INK, lw=2.0, zorder=6)
            ax.text((a2 + z2)/2, Y["cond"]*H, cond, color=INK, fontsize=17, fontweight="bold",
                    ha="center", va="center", zorder=6)
    for i, x in enumerate(lx):
        pl = DIL[i] is None
        ax.text(x, Y["dil"]*H, "P.L" if pl else DIL[i], color="#374151", fontsize=15.5 if pl else 15,
                fontweight="bold" if pl else "normal", ha="center", va="center", zorder=6)
        ax.add_patch(plt.Circle((x, Y["num"]*H), step*.185, fc=GREY if pl else "#111111",
                                ec="none", zorder=6))
        ax.text(x, Y["num"]*H, str(i + 1), color="white", fontsize=15, fontweight="bold",
                ha="center", va="center", zorder=7)
    ax.text(-SIDE*.94, Y["num"]*H, "lane", color=INK, fontsize=13, fontweight="bold",
            ha="left", va="center", zorder=6)

    for b in c["blocks"]:
        i0, i1 = b["lanes"]; fs = np.linspace(i0 - .46, i1 + .46, 40)/(NL-1)
        top = np.array([y_of(fn, b["mw"]*TOL, f) for f in fs])
        bot = np.array([y_of(fn, b["mw"]/TOL, f) for f in fs])
        xt  = np.array([lane_x(fn, f*(NL-1), y) for f, y in zip(fs, top)])
        xb  = np.array([lane_x(fn, f*(NL-1), y) for f, y in zip(fs, bot)])
        ring = np.c_[np.r_[xt, xb[::-1]], np.r_[top, bot[::-1]]]
        ax.add_patch(Polygon(ring, closed=True, fc=RED, alpha=0.07, ec="none", zorder=4))
        ax.add_patch(Polygon(ring, closed=True, fc="none", ec=RED, lw=3.2, joinstyle="miter", zorder=5))
        t0, t1 = b["tag"]; xm = (lane_x(fn, t0, bot.mean()) + lane_x(fn, t1, bot.mean()))/2
        ax.text(xm, bot.max() + 0.012*H, f"{b['mw']:.1f} kDa ±9%", color=RED, fontsize=13,
                fontweight="bold", ha="center", va="top", zorder=6,
                bbox=dict(fc="white", ec=RED, lw=1.5, boxstyle="round,pad=0.26"))

    for _, r in tables[fn].iterrows():
        if r["sample"] == "P.L" or not r["detected"]: continue
        i = int(r["lane"]) - 1; y = r["band_y"]; x = lane_x(fn, i, y) + pitch(fn, y)*.47
        s = step*.13
        ax.add_patch(Polygon([[x, y], [x + s, y - s*.62], [x + s, y + s*.62]], closed=True,
                             fc=INK, ec="white", lw=1.0, zorder=8))
        ax.text(x + s*.2, y - s*.95, f"{r['band_kDa_piecewise']:.0f}–{r['band_kDa_cubic']:.0f}",
                color=INK, fontsize=11.5, fontweight="bold", ha="center", va="bottom", zorder=8,
                bbox=dict(fc="white", ec="none", alpha=.78, boxstyle="round,pad=0.12"))

    ax.set_xlim(-SIDE, W + SIDE); ax.set_ylim(H, -HEAD); ax.axis("off")
    fig.text(.5, .985, f"{c['gel']}  ·  2026-09-29 SDS-PAGE, Coomassie R-250  ·  ladder: PageRuler "
             f"Prestained, 170–10 kDa (70 kDa orange, 10 kDa green)",
             fontsize=15, fontweight="bold", ha="center", va="top")
    fig.text(.5, .952, "red box = where that construct's band should fall, widened by the ±9 % ladder "
             "calibration error   ·   ◀ = strongest band found in the lane at 30–100 kDa, apparent kDa "
             "(rung-to-rung – cubic fit)",
             fontsize=12, color=GREY, ha="center", va="top")
    foot = ("background-flattened for visibility: each channel divided by a 400 px, 80th-percentile "
            "estimate of the clear-gel background" if corr == "flattened" else
            "uniform linear contrast stretch only — no background correction")
    if palette == "color":
        foot += (f"; saturation lifted {SAT:.1f}x globally" if corr == "flattened"
                 else "; the same stretch on all three channels, colours as photographed")
    foot += ". No local or per-lane edits."
    fig.text(.5, .006, foot, fontsize=11.5, color=GREY, ha="center", va="bottom", style="italic")

    p = OUT/f"{fn.replace('.jpeg','')}_annotated_{corr}_{palette}.jpg"
    fig.savefig(p, dpi=120, bbox_inches="tight", facecolor="white", pil_kwargs={"quality": 92}); plt.close(fig)
    return p


for fn in BOX:
    for corr in ("uniform", "flattened"):
        for palette in ("gray", "color"):
            print("wrote", draw(fn, corr, palette, OUT, tables).relative_to(ROOT))

wrote code/output/0929_gel_annotation/cds12-cds11_annotated_uniform_gray.jpg


wrote code/output/0929_gel_annotation/cds12-cds11_annotated_uniform_color.jpg


wrote code/output/0929_gel_annotation/cds12-cds11_annotated_flattened_gray.jpg


wrote code/output/0929_gel_annotation/cds12-cds11_annotated_flattened_color.jpg


wrote code/output/0929_gel_annotation/e281-e1_annotated_uniform_gray.jpg


wrote code/output/0929_gel_annotation/e281-e1_annotated_uniform_color.jpg


wrote code/output/0929_gel_annotation/e281-e1_annotated_flattened_gray.jpg


wrote code/output/0929_gel_annotation/e281-e1_annotated_flattened_color.jpg


## 8. Save the measurements

In [9]:
for fn, df in tables.items():
    p = OUT/f"{fn.replace('.jpeg','')}_lane_quant.tsv"
    df.to_csv(p, sep="\t", index=False); print(p.relative_to(ROOT))

cal = pd.DataFrame([dict(gel=fn[:-5], band=i+1, mw_kDa=m, y_left=yl, y_right=yr)
                    for fn in BOX
                    for i, (m, yl, yr) in enumerate(zip(LADDER_MW, CAL[fn]["yL"], CAL[fn]["yR"]))])
p = OUT/"ladder_calibration.tsv"; cal.to_csv(p, sep="\t", index=False); print(p.relative_to(ROOT))

code/output/0929_gel_annotation/cds12-cds11_lane_quant.tsv
code/output/0929_gel_annotation/e281-e1_lane_quant.tsv
code/output/0929_gel_annotation/ladder_calibration.tsv


---
*Log — 2026-09-29: first pass on the two 2026-09-29 expression gels, same pipeline as 0915. Ladder
is the same PageRuler Prestained (10 rungs, 4th red, 10th green, on all four ladder lanes). Three
method changes, each forced by the photographs: (1) background is an 80th-percentile estimate in a
400 px window instead of a 70 px Gaussian, because raw lanes here are full-height smears; signal is
optical density; (2) each rung is interpolated between its left and right ladder position instead of
one global tilt, and lanes follow the flare of the gel; (3) a rung-to-rung calibration is reported
beside the 0915 cubic, since they differ by 2–3 kDa between the 55 and 40 kDa rungs. Added to the
table and figures: the strongest 30–100 kDa band per lane, with a noise floor of 0.055 set by the
paper texture under the tray. Result: CDS12_WT shows an IPTG-only band at 40–43 kDa on the predicted
41.8; e1 shows a strong band at 45–50 kDa in all four lanes regardless of induction; CDS11 shows
nothing at 72 kDa; e281 shows nothing at 42.3. Open items: (1) empty-vector / untransformed-BL21
lane — UNDONE; (2) anti-His western or Ni-NTA — UNDONE; (3) raw lanes for CDS11 and e1 — UNDONE;
(4) the label sheets leave date, gel %, run conditions and all sample volumes blank; (5) whether
"extract" is whole-cell or cleared lysate is not recorded; (6) a ~48–50 kDa band occupies wells 6–9
of gel 1 across the construct boundary and is unexplained. Inputs were copied from `sds_gel_0929/`
and renamed; the originals are untouched.*